<a href="https://colab.research.google.com/github/replyeshab/CineAI-AI-Based-Hybrid-Recommendation-System/blob/main/hybrid.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Hybrid Recommendation

In [1]:
import numpy as np
import pandas as pd
import pickle

from sklearn.preprocessing import MinMaxScaler

In [2]:
import joblib

ARTIFACT_PATH = "/content/drive/MyDrive/CineAI/Artifact"

movie_data = joblib.load(ARTIFACT_PATH+"/movie_data.pkl")

popular_movies = joblib.load(ARTIFACT_PATH+"/popular_movies.pkl")

svd_model = joblib.load(ARTIFACT_PATH+"/svd_model.pkl")

knn_model = joblib.load(ARTIFACT_PATH+"/knn_content_model.pkl")

tfidf_matrix = joblib.load(ARTIFACT_PATH+"/tfidf_matrix.pkl")

tfidf_vectorizer = joblib.load(ARTIFACT_PATH+"/tfidf_vectorizer.pkl")

title_lookup = joblib.load(ARTIFACT_PATH+"/title_lookup.pkl")

title_to_index = joblib.load(ARTIFACT_PATH+"/title_to_index.pkl")

user_encoder = joblib.load(ARTIFACT_PATH+"/user_encoder.pkl")

movie_encoder = joblib.load(ARTIFACT_PATH+"/movie_encoder.pkl")

user_features = joblib.load(ARTIFACT_PATH + "/user_features.pkl")
movie_features = joblib.load(ARTIFACT_PATH + "/movie_features.pkl")
train_df = pd.read_pickle(
    ARTIFACT_PATH + "/train_df.pkl"
)

val_df = pd.read_pickle(
    ARTIFACT_PATH + "/val_df.pkl"
)

test_df = pd.read_pickle(
    ARTIFACT_PATH + "/test_df.pkl"
)


In [3]:
ratings = pd.read_csv(
    "/content/drive/MyDrive/CineAI/ml-32m/ml-32m/ratings.csv"
)

In [4]:
movie_stats = pd.read_pickle("/content/drive/MyDrive/CineAI/Artifact/movie_stats.pkl")

# Popularity Recommendation

In [5]:
def calculate_imdb_score(df, quantile=0.90):

    df = df.copy()

    C = df["average_rating"].mean()

    m = df["rating_count"].quantile(quantile)

    df["weighted_rating"] = (
        (
            df["rating_count"] /
            (df["rating_count"] + m)
        ) * df["average_rating"]
    ) + (
        (
            m /
            (df["rating_count"] + m)
        ) * C
    )

    return df

In [6]:
popular_movies = calculate_imdb_score(movie_stats)

In [7]:
popular_movies = (
    popular_movies
    .sort_values(
        by="weighted_rating",
        ascending=False
    )
    .reset_index(drop=True)
)

In [8]:
def recommend_popular(top_n=500):

    columns = [
        "movieId",
        "clean_title",
        "genres",
        "average_rating",
        "rating_count",
        "weighted_rating"
    ]

    return popular_movies[columns].head(top_n)

In [9]:
popular = recommend_popular(20)
popular.head()

,movieId,clean_title,genres,average_rating,rating_count,weighted_rating
0,318,"Shawshank Redemption, The",Crime|Drama,4.404614,102929,4.401238
1,159817,Planet Earth,Documentary,4.444369,2948,4.332311
2,858,"Godfather, The",Crime|Drama,4.317030,66440,4.312134
3,170705,Band of Brothers,Action|Drama|War,4.426539,2811,4.310914
4,202439,Parasite,Comedy|Drama,4.312254,11670,4.284956


# Cold Start

In [10]:
def recommend_cold_start(popular_df, top_n=20):


    cold_start = (
        popular_df
        .sort_values(
            by=[
                "weighted_rating",
                "rating_count",
                "average_rating"
            ],
            ascending=False
        )
        .head(top_n)
    )

    return cold_start

In [11]:
USER_ID = 1
def is_new_user(user_id, ratings):

    if user_id is None:
        return True

    if user_id not in ratings["userId"].values:
        return True

    return False

def is_sparse_user(
    user_id,
    ratings,
    threshold=10
):

    if is_new_user(user_id, ratings):
        return False

    n = len(
        ratings[
            ratings["userId"] == user_id
        ]
    )

    return n < threshold

In [12]:
if is_new_user(USER_ID, ratings):

    print("Cold Start User")

    final_recommendations = recommend_cold_start(
        popular_movies,
        top_n=500
    )

    display(final_recommendations)

    raise SystemExit()

# Collaborative Recommendation

In [13]:
def rank_candidates(
    candidate_indices,
    scores,
    top_n=500
):

    top_indices = candidate_indices[:top_n]

    movie_ids = movie_encoder.inverse_transform(
        top_indices
    )

    recommendations = (
        movie_data[
            movie_data.movieId.isin(
                movie_ids
            )
        ]
        .copy()
    )

    score_map = {
        movie_encoder.inverse_transform([idx])[0]:
        scores[idx]
        for idx in top_indices
    }

    recommendations["svd_score"] = (
        recommendations.movieId
        .map(score_map)
    )

    return recommendations.sort_values(
        "svd_score",
        ascending=False
    )

In [14]:
def filter_seen_movies(
    user_id,
    candidate_indices,
    scores,
    ratings_df
):

    watched = ratings_df.loc[
        ratings_df.userId == user_id,
        "movieId"
    ]

    watched = watched[
        watched.isin(
            movie_encoder.classes_
        )
    ]

    watched_indices = set(
        movie_encoder.transform(
            watched
        )
    )

    filtered = [
        idx
        for idx in candidate_indices
        if idx not in watched_indices
    ]

    return filtered

In [15]:
def generate_candidates(
    user_id,
    candidate_size=500
):

    if user_id not in user_encoder.classes_:
        raise ValueError("Unknown User ID")

    user_index = user_encoder.transform(
        [user_id]
    )[0]

    user_vector = user_features[user_index]

    scores = movie_features @ user_vector

    top_candidates = np.argsort(
        scores
    )[::-1][:candidate_size]

    return top_candidates, scores

In [16]:
def recommend_for_user_collaborative(
    user_id,
    ratings_df, # Added ratings_df parameter
    top_n=500
):

    candidates, scores = generate_candidates(
        user_id
    )

    filtered_candidates = filter_seen_movies(
      user_id,
      candidates,
      scores,
      ratings_df
)

    recommendations = rank_candidates(
        filtered_candidates,
        scores,
        top_n
    )

    recommendations = (
        recommendations
        .sort_values("svd_score", ascending=False)
        .reset_index(drop=True)
    )

    recommendations.index.name = "Rank"

    return recommendations

In [17]:
collab = recommend_for_user_collaborative(user_id=1, ratings_df=ratings, top_n=20)
collab.head()

,movieId,title,genres,year,clean_title,genres_list,imdbId,tmdbId,tag,combined_features,svd_score
Rank,,,,,,,,,,,
0,858,"Godfather, The (1972)",Crime|Drama,1972.0,"Godfather, The","[Crime, Drama]",68646,238.0,al pacino atmospheric great acting masterpiece...,"Godfather, The Crime Drama Al Pacino atmospher...",2.863783
1,1222,Full Metal Jacket (1987),Drama|War,1987.0,Full Metal Jacket,"[Drama, War]",93058,600.0,vietnam war tumey s dvds imdb top 250 stanley ...,Full Metal Jacket Drama War Vietnam War Tumey'...,2.771993
2,924,2001: A Space Odyssey (1968),Adventure|Drama|Sci-Fi,1968.0,2001: A Space Odyssey,"[Adventure, Drama, Sci-Fi]",62622,62.0,atmospheric space stanley kubrick ambiguous at...,2001: A Space Odyssey Adventure Drama Sci-Fi a...,2.545833
3,750,Dr. Strangelove or: How I Learned to Stop Worr...,Comedy|War,1964.0,Dr. Strangelove or: How I Learned to Stop Worr...,"[Comedy, War]",57012,935.0,black and white stanley kubrick classic dark c...,Dr. Strangelove or: How I Learned to Stop Worr...,2.479119
4,2858,American Beauty (1999),Drama|Romance,1999.0,American Beauty,"[Drama, Romance]",169547,14.0,bittersweet great acting kevin spacey loneline...,American Beauty Drama Romance bittersweet grea...,2.291048


# Content Recommedation

In [18]:
from sklearn.metrics.pairwise import linear_kernel
def recommend_content(title, top_n=500):

    matches = movie_data[
        movie_data["clean_title"]
        .str.lower() == title.lower()
    ]

    if matches.empty:
        print("Movie not found.")
        return pd.DataFrame(
          columns=[
             "movieId",
            "similarity_score"
    ]
)

    idx = matches.index[0]

    cosine_scores = linear_kernel(
        tfidf_matrix[idx:idx + 1],
        tfidf_matrix
    ).flatten()

    similar_indices = np.argsort(cosine_scores)[::-1]

    similar_indices = similar_indices[1:top_n + 1]

    recommendations = movie_data.iloc[
        similar_indices
    ][["movieId", "clean_title", "genres", "year"]].copy()

    recommendations["similarity_score"] = (
        cosine_scores[similar_indices]
    )

    return recommendations

In [19]:
def generate_candidate_pool(
    user_id,
    ratings_df,
    reference_movie=None,
    top_n=500
):
  collab = recommend_for_user_collaborative(
    user_id=user_id,
    ratings_df=ratings_df,
    top_n=top_n
)

  popular = recommend_popular(
      top_n
)

  if reference_movie is not None:

    content = recommend_content(
        reference_movie,
        top_n
    )

  else:

    watched = ratings_df.loc[
        ratings_df["userId"] == user_id,
        "movieId"
    ]

    if len(watched) == 0:

        content = pd.DataFrame(
            columns=[
                "movieId",
                "similarity_score"
            ]
        )

    else:

        user_history = ratings_df[
        ratings_df["userId"] == user_id
        ].sort_values("timestamp")

        latest_movie = user_history.iloc[-1]["movieId"]

        movie_row = movie_data[
            movie_data["movieId"] == latest_movie
        ]

        if movie_row.empty:

            content = pd.DataFrame(
                columns=[
                    "movieId",
                    "similarity_score"
                ]
            )

        else:

            title = movie_row.iloc[0]["clean_title"]

            content = recommend_content(
                title,
                top_n
            )
  collab = (
    collab[
        [
            "movieId",
            "svd_score"
        ]
    ]
    .rename(
        columns={
            "svd_score":"collab_score"
        }
    )
)
  content = (
    content[
        [
            "movieId",
            "similarity_score"
        ]
    ]
    .rename(
        columns={
            "similarity_score":"content_score"
        }
    )
)
  popular = (
    popular[
        [
            "movieId",
            "average_rating",
            "rating_count",
            "weighted_rating"
        ]
    ]
    .rename(
        columns={
            "weighted_rating":"popularity_score"
        }
    )
)
  hybrid = (
    collab
    .merge(
        content,
        on="movieId",
        how="outer"
    )
    .merge(
        popular,
        on="movieId",
        how="outer"
    )
)
  hybrid = hybrid.drop_duplicates("movieId")
  score_columns = [
    "collab_score",
    "content_score",
    "popularity_score"
]

  for col in score_columns:

    hybrid[col] = scaler.fit_transform(
        hybrid[[col]].fillna(0)
    )
  return hybrid

In [20]:
content = recommend_content("Toy Story", 20)
content.head()

,movieId,clean_title,genres,year,similarity_score
3021,3114,Toy Story 2,Adventure|Animation|Children|Comedy|Fantasy,1999.0,0.892434
2264,2355,"Bug's Life, A",Adventure|Animation|Children|Comedy,1998.0,0.806176
14815,78499,Toy Story 3,Adventure|Animation|Children|Comedy|Fantasy|IMAX,2010.0,0.711669
39850,157296,Finding Dory,Adventure|Animation|Comedy,2016.0,0.701200
4781,4886,"Monsters, Inc.",Adventure|Animation|Children|Comedy|Fantasy,2001.0,0.700583


In [21]:
print(collab.columns)
print(content.columns)
print(popular.columns)

Index(['movieId', 'title', 'genres', 'year', 'clean_title', 'genres_list',
       'imdbId', 'tmdbId', 'tag', 'combined_features', 'svd_score'],
      dtype='object')
Index(['movieId', 'clean_title', 'genres', 'year', 'similarity_score'], dtype='object')
Index(['movieId', 'clean_title', 'genres', 'average_rating', 'rating_count',
       'weighted_rating'],
      dtype='object')


In [22]:
print(collab.head())
print(content.head())
print(popular.head())

      movieId                                              title  \
Rank                                                               
0         858                              Godfather, The (1972)   
1        1222                           Full Metal Jacket (1987)   
2         924                       2001: A Space Odyssey (1968)   
3         750  Dr. Strangelove or: How I Learned to Stop Worr...   
4        2858                             American Beauty (1999)   

                      genres    year  \
Rank                                   
0                Crime|Drama  1972.0   
1                  Drama|War  1987.0   
2     Adventure|Drama|Sci-Fi  1968.0   
3                 Comedy|War  1964.0   
4              Drama|Romance  1999.0   

                                            clean_title  \
Rank                                                      
0                                        Godfather, The   
1                                     Full Metal Jacket   
2     

# Genre Recommedation

In [23]:
def build_genre_profile(
    user_id,
    ratings,
    movie_data
):


    user_ratings = ratings[
        ratings["userId"] == user_id
    ]

    profile = {}

    user_movies = user_ratings.merge(
        movie_data[
            [
                "movieId",
                "genres"
            ]
        ],
        on="movieId",
        how="left"
    )

    for _, row in user_movies.iterrows():

        if pd.isna(row["genres"]):
            continue

        genres = row["genres"].split("|")

        rating = row["rating"]

        for genre in genres:

            if genre == "(no genres listed)":
                continue

            profile[genre] = (
                profile.get(genre, 0)
                + rating
            )

    if len(profile) == 0:
        return {}

    maximum = max(profile.values())

    for genre in profile:

        profile[genre] /= maximum

    return profile

In [24]:
def calculate_genre_score(hybrid_df, genre_profile):

    genre_scores = []

    for _, row in hybrid_df.iterrows():

        if pd.isna(row["genres"]):
            genre_scores.append(0)
            continue

        movie_genres = row["genres"].split("|")

        score = 0

        count = 0

        for genre in movie_genres:

            if genre == "(no genres listed)":
                continue

            score += genre_profile.get(genre, 0)

            count += 1

        if count == 0:
            genre_scores.append(0)

        else:
            genre_scores.append(score / count)

    hybrid_df["genre_score"] = genre_scores

    return hybrid_df

In [25]:
import datetime

def build_features(
    df,
    user_id,
    ratings_df
):
  df["rating_score"] = scaler.fit_transform(
    df[
        ["average_rating"]
    ].fillna(0)
)
  df["rating_count_score"] = scaler.fit_transform(
    df[
        ["rating_count"]
    ].fillna(0)
)
  df = df.merge(
    movie_data[
        ["movieId", "year","genres"]
    ],
    on="movieId",
    how="left"
)
  watched_movies = ratings_df.loc[
    ratings_df["userId"] == user_id,
    "movieId"
]

  df = df[
    ~df["movieId"].isin(
        watched_movies
    )
]
  genre_profile = build_genre_profile(
    user_id,
    ratings_df,
    movie_data
)

  df = calculate_genre_score(
    df,
    genre_profile
)
  df["genre_score"] = scaler.fit_transform(
    df[
        ["genre_score"]
    ].fillna(0)
)
  new_movie_mask = (
    df["rating_count"].fillna(0) == 0
)

  df.loc[
    new_movie_mask,
    "collab_score"
] = np.nan

  df.loc[
    new_movie_mask,
    "popularity_score"
] = np.nan

  CURRENT_YEAR = datetime.datetime.now().year

  df["movie_age"] = CURRENT_YEAR - df["year"]

  df["recency_score"] = 1 / (1 + df["movie_age"])

  df["recency_score"] = scaler.fit_transform(
    df[["recency_score"]]
)
  return df

In [26]:
def get_dynamic_weights(
    user_id,
    ratings
):

    if is_new_user(
        user_id,
        ratings
    ):

        return {
            "collab_score":0,
            "content_score":0,
            "popularity_score":0.45,
            "genre_score":0,
            "rating_score":0.25,
            "rating_count_score":0.20,
            "recency_score":0.10
        }

    if is_sparse_user(
        user_id,
        ratings
    ):

        return {
            "collab_score":0.10,
            "content_score":0.40,
            "popularity_score":0.15,
            "genre_score":0.25,
            "rating_score":0.05,
            "rating_count_score":0.03,
            "recency_score":0.02
        }

    return {
        "collab_score":0.20,
        "content_score":0.30,
        "popularity_score":0.15,
        "genre_score":0.20,
        "rating_score":0.05,
        "rating_count_score":0.05,
        "recency_score":0.05
    }

In [27]:
score_columns = [
    "collab_score",
    "content_score",
    "popularity_score",
    "genre_score",
    "rating_score",
    "rating_count_score",
    "movie_age"
]

In [28]:
def rank_movies(
    df,
    user_id,
    ratings
):

    weights = get_dynamic_weights(
        user_id,
        ratings
    )

    final_scores = []

    for _, row in df.iterrows():

        score = 0
        total_weight = 0

        for feature, weight in weights.items():

            value = row[feature]

            if pd.notna(value):

                score += value * weight

                total_weight += weight

        if total_weight > 0:

            score /= max(total_weight, 1e-8)

        final_scores.append(score)

    df["final_score"] = final_scores

    return df

In [29]:
def validate_recommendations(
    recommendations,
    watched_movies
):

    assert (
        recommendations["movieId"]
        .duplicated()
        .sum() == 0
    ), "Duplicate recommendations"

    assert (
        len(
            set(
                recommendations["movieId"]
            ).intersection(
                watched_movies
            )
        ) == 0
    ), "Already watched movie recommended"

    assert (
        recommendations["final_score"]
        .isna()
        .sum() == 0
    ), "NaN score detected"

    print("All validation checks passed.")

In [30]:
def get_final_recommendations(
    df,
    top_n=500
):
    df = df.sort_values(
        "final_score",
        ascending=False
    )

    return df.head(top_n)

In [31]:
def recommend_movies(
    user_id,
    reference_movie=None,
    ratings_df=None,
    top_n=500
):


    if is_new_user(user_id, ratings_df):

        print("New User Detected")

        return recommend_cold_start(
            popular_movies,
            top_n=top_n
        )


    candidate_pool = generate_candidate_pool(
    user_id=user_id,
    ratings_df=ratings_df,
    reference_movie=reference_movie,
    top_n=top_n
)
    if candidate_pool.empty:
      print("Candidate pool empty. Falling back to popularity.")
      return recommend_popular(top_n)

    candidate_pool = build_features(
        df=candidate_pool,
        user_id=user_id,
        ratings_df=ratings_df
)



    candidate_pool = rank_movies(
        df=candidate_pool,
        user_id=user_id,
        ratings=ratings_df
    )



    recommendations = get_final_recommendations(
        candidate_pool,
        top_n=top_n
    )



    watched_movies = ratings_df.loc[
    ratings_df["userId"] == user_id,
    "movieId"
]

    validate_recommendations(
        recommendations,
        watched_movies
    )

    return recommendations

In [32]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()

final_recommendations = recommend_movies(
    user_id=1,
    reference_movie="Toy Story",
    ratings_df=train_df,
    top_n=20
)

display(final_recommendations)

All validation checks passed.


,movieId,collab_score,content_score,average_rating,rating_count,popularity_score,rating_score,rating_count_score,year,genres,genre_score,movie_age,recency_score,final_score
32,3114,NaN,1.000000,NaN,NaN,NaN,0.000000,0.000000,1999.0,Adventure|Animation|Children|Comedy|Fantasy,0.154864,27.0,0.212454,0.525531
8,858,0.746988,0.000000,4.317030,66440.0,0.979755,0.970811,0.645493,1972.0,Crime|Drama,0.575875,54.0,0.057809,0.495242
29,2355,NaN,0.903345,NaN,NaN,NaN,0.000000,0.000000,1998.0,Adventure|Animation|Children|Comedy,0.179961,28.0,0.201592,0.487808
50,157296,NaN,0.785717,NaN,NaN,NaN,0.000000,0.000000,2016.0,Adventure|Animation|Comedy,0.230869,10.0,0.699301,0.487467
39,50872,NaN,0.717347,NaN,NaN,NaN,0.000000,0.000000,2007.0,Animation|Children|Drama,0.342412,19.0,0.338462,0.462476
19,1221,0.620139,0.000000,4.264468,43111.0,0.967282,0.958991,0.418842,1974.0,Crime|Drama,0.575875,52.0,0.063861,0.456380
47,103141,NaN,0.740759,NaN,NaN,NaN,0.000000,0.000000,2013.0,Adventure|Animation|Comedy,0.230869,13.0,0.527473,0.453500
42,78499,NaN,0.797447,NaN,NaN,NaN,0.000000,0.000000,2010.0,Adventure|Animation|Children|Comedy|Fantasy|IMAX,0.129053,16.0,0.416290,0.439784
35,6377,NaN,0.780009,NaN,NaN,NaN,0.000000,0.000000,2003.0,Adventure|Animation|Children|Comedy,0.179961,23.0,0.264957,0.435758
36,8961,NaN,0.767647,NaN,NaN,NaN,0.000000,0.000000,2004.0,Action|Adventure|Animation|Children|Comedy,0.184436,22.0,0.280936,0.432658


# Evaluation Matrix

In [33]:
from sklearn.metrics import ndcg_score
from sklearn.model_selection import train_test_split

In [34]:
def precision_at_k(recommended_items, relevant_items, k=10):

    recommended_items = recommended_items[:k]

    hits = len(set(recommended_items) & set(relevant_items))

    return hits / k if k > 0 else 0

In [35]:
def recall_at_k(recommended_items, relevant_items, k=10):


    recommended_items = recommended_items[:k]

    hits = len(set(recommended_items) & set(relevant_items))

    if len(relevant_items) == 0:
        return 0

    return hits / len(relevant_items)

In [36]:
def average_precision(recommended_items, relevant_items, k=10):


    recommended_items = recommended_items[:k]

    score = 0.0
    hits = 0

    for i, movie in enumerate(recommended_items, start=1):

        if movie in relevant_items:
            hits += 1
            score += hits / i

    if len(relevant_items) == 0:
        return 0

    return score / min(len(relevant_items), k)

In [37]:
def mean_average_precision(all_recommendations,
                           ground_truth,
                           k=10):

    scores = []

    for user in all_recommendations.keys():

        ap = average_precision(
            all_recommendations[user],
            ground_truth[user],
            k
        )

        scores.append(ap)

    return np.mean(scores)

In [38]:
from sklearn.metrics import ndcg_score
def ndcg_at_k(recommended_items,
              relevant_items,
              k=10):

    recommended_items = recommended_items[:k]

    y_true = [
        1 if movie in relevant_items else 0
        for movie in recommended_items
    ]

    y_score = list(range(k, 0, -1))

    return ndcg_score([y_true], [y_score])

In [39]:
def coverage(all_recommendations,
             total_movies):

    recommended_movies = set()

    for movies in all_recommendations.values():
        recommended_movies.update(movies)

    return len(recommended_movies) / total_movies

In [40]:
import time

def inference_latency(recommendation_function,
                      *args,
                      **kwargs):

    start = time.time()

    recommendation_function(*args, **kwargs)

    end = time.time()

    return end - start

In [41]:
def novelty(all_recommendations, train_df):

    # Number of interactions for each movie
    movie_popularity = (
        train_df
        .groupby("movieId")
        .size()
    )

    # Convert popularity to probability
    movie_probability = (
        movie_popularity /
        movie_popularity.sum()
    )

    novelty_scores = []

    for recommended_movies in all_recommendations.values():

        for movie_id in recommended_movies:

            if movie_id in movie_probability.index:

                probability = movie_probability.loc[movie_id]

                # Avoid log(0)
                probability = max(probability, 1e-12)

                novelty_scores.append(
                    -np.log2(probability)
                )

    if len(novelty_scores) == 0:
        return 0.0

    return np.mean(novelty_scores)

In [42]:
def diversity(all_recommendations, movies):
    """
    Calculate average recommendation diversity using genre dissimilarity.
    """

    movie_genres = (
        movies.set_index("movieId")["genres"]
        .fillna("")
        .to_dict()
    )

    diversity_scores = []

    for recommended_movies in all_recommendations.values():

        genre_sets = []

        for movie_id in recommended_movies:

            genres = movie_genres.get(movie_id, "")

            genre_sets.append(
                set(genres.split("|"))
            )

        n = len(genre_sets)

        if n < 2:
            continue

        pairwise_diversity = []

        for i in range(n):

            for j in range(i + 1, n):

                intersection = len(
                    genre_sets[i] & genre_sets[j]
                )

                union = len(
                    genre_sets[i] | genre_sets[j]
                )

                if union == 0:
                    similarity = 0
                else:
                    similarity = intersection / union

                pairwise_diversity.append(
                    1 - similarity
                )

        if pairwise_diversity:
            diversity_scores.append(
                np.mean(pairwise_diversity)
            )

    if len(diversity_scores) == 0:
        return 0.0

    return np.mean(diversity_scores)

In [43]:
def hit_rate_at_k(recommended_items,
                  relevant_items,
                  k=10):

    recommended_items = recommended_items[:k]

    return int(
        len(
            set(recommended_items)
            &
            set(relevant_items)
        ) > 0
    )

In [44]:
def evaluate_recommender(
    train_df,
    test_df,
    movies,
    ratings,
    top_k=10
):
    """
    Evaluate CineAI recommender.
    """

    precision_scores = []
    recall_scores = []
    map_scores = []
    ndcg_scores = []
    hit_rate_scores = []
    all_recommendations = {}
    ground_truth = {}

    users = test_df["userId"].unique()

    for user_id in users:

        try:

            # Generate recommendations using ONLY train data
            recommendations = recommend_movies(
                  user_id=user_id,
                  reference_movie=None,
                  ratings_df=train_df,
                  top_n=top_k
)

            if recommendations is None or len(recommendations) == 0:
                continue

            recommended_items = recommendations["movieId"].tolist()

            relevant_items = test_df[
                test_df["userId"] == user_id
            ]["movieId"].tolist()

            all_recommendations[user_id] = recommended_items
            ground_truth[user_id] = relevant_items

            precision_scores.append(
                precision_at_k(
                    recommended_items,
                    relevant_items,
                    top_k
                )
            )

            recall_scores.append(
                recall_at_k(
                    recommended_items,
                    relevant_items,
                    top_k
                )
            )
            hit_rate_scores.append(
                hit_rate_at_k(
                  recommended_items,
                  relevant_items,
                  top_k
    )
)
            map_scores.append(
                average_precision(
                    recommended_items,
                    relevant_items,
                    top_k
                )
            )

            ndcg_scores.append(
                ndcg_at_k(
                    recommended_items,
                    relevant_items,
                    top_k
                )
            )

        except Exception as e:
            print(f"Skipping User {user_id}: {e}")

    coverage_score = coverage(
        all_recommendations,
        movies["movieId"].nunique()
    )

    diversity_score = diversity(
        all_recommendations,
        movies
    )

    novelty_score = novelty(
        all_recommendations,
        train_df
    )

    latency = inference_latency(
    recommend_movies,
    user_id=users[0],
    ratings_df=train_df,
    reference_movie=None,
    top_n=top_k
)

    print("=" * 45)
    print("         CineAI Evaluation Report")
    print("=" * 45)

    print(f"Users Evaluated : {len(all_recommendations)}")

    print(f"\nPrecision@{top_k} : {np.mean(precision_scores):.4f}")
    print(f"Recall@{top_k}    : {np.mean(recall_scores):.4f}")
    print(f"MAP@{top_k}       : {np.mean(map_scores):.4f}")
    print(f"NDCG@{top_k}      : {np.mean(ndcg_scores):.4f}")

    print(f"\nCoverage          : {coverage_score:.4f}")
    print(f"Diversity           : {diversity_score:.4f}")
    print(f"Novelty             : {novelty_score:.4f}")
    print(f"HitRate@{top_k}     : {np.mean(hit_rate_scores):.4f}")
    print(f"\nLatency (sec)     : {latency:.4f}")

    print("=" * 45)

    return {
        "precision": np.mean(precision_scores),
        "recall": np.mean(recall_scores),
        "map": np.mean(map_scores),
        "ndcg": np.mean(ndcg_scores),
        "coverage": coverage_score,
        "diversity": diversity_score,
        "novelty": novelty_score,
        "latency": latency,
        "hit_rate": np.mean(hit_rate_scores),
    }

In [ ]:
results = evaluate_recommender(train_df,
    test_df,
    movie_data,
    ratings,
    500)

All validation checks passed.
All validation checks passed.
All validation checks passed.
All validation checks passed.
All validation checks passed.
All validation checks passed.
All validation checks passed.
All validation checks passed.
All validation checks passed.
All validation checks passed.
All validation checks passed.
All validation checks passed.
All validation checks passed.
All validation checks passed.
All validation checks passed.
All validation checks passed.
All validation checks passed.
All validation checks passed.
All validation checks passed.
All validation checks passed.
All validation checks passed.
All validation checks passed.
All validation checks passed.
All validation checks passed.
All validation checks passed.
All validation checks passed.
All validation checks passed.
All validation checks passed.
All validation checks passed.
All validation checks passed.
All validation checks passed.
All validation checks passed.
All validation checks passed.
All valida